# 金安国纪价格技术指标分析

本 notebook 使用 task1 已经保存的金安国纪日线行情数据，不重新调用 Tushare。计算 RSI、MACD、布林带和 ATR，并展示计算过程、图表和最近交易日摘要。

In [ ]:
from pathlib import Path
import sys
import pandas as pd

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'task1').exists() and (candidate / 'task2').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录：需要同时包含 task1 和 task2')

ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(ROOT))

from task2.src.indicators import clean_price_data, add_all_indicators, latest_indicator_summary
from task2.src.reporting import write_all_svg_charts, write_summary_markdown

INPUT_CSV = ROOT / 'task1' / 'outputs' / 'jinan_guoji' / 'daily_prices.csv'
DATA_CSV = ROOT / 'task2' / 'data' / 'jinan_guoji_indicators.csv'
SUMMARY_MD = ROOT / 'task2' / 'outputs' / 'jinan_guoji_indicator_summary.md'
FIGURE_DIR = ROOT / 'task2' / 'outputs' / 'figures'
INPUT_CSV

## 1. 读取原始数据

原始数据来自 task1，不在本任务中重新拉取。

In [ ]:
raw = pd.read_csv(INPUT_CSV)
print(raw.shape)
raw.head()

## 2. 清洗和排序

将交易日期转为日期类型，核心价格和成交量字段转为数值，并按交易日期升序排列。

In [ ]:
prices = clean_price_data(raw)
print(prices[['trade_date', 'open', 'high', 'low', 'close', 'vol']].dtypes)
prices.head()

## 3. 数据质量检查

检查缺失值、重复交易日和价格字段的基本合理性。

In [ ]:
required = ['trade_date', 'open', 'high', 'low', 'close', 'vol']
quality = pd.DataFrame({
    'missing_count': prices[required].isna().sum(),
})
duplicate_dates = prices['trade_date'].duplicated().sum()
invalid_price_rows = ((prices['high'] < prices['low']) | (prices['close'] <= 0)).sum()
print(f'重复交易日数量: {duplicate_dates}')
print(f'价格异常行数量: {invalid_price_rows}')
quality

## 4. 计算技术指标

- RSI 使用 14 日 Wilder 平滑。
- MACD 使用 12、26、9 参数。
- 布林带使用 20 日均线和 2 倍标准差。
- ATR 使用 14 日真实波幅 Wilder 平滑。

In [ ]:
indicators = add_all_indicators(prices)
DATA_CSV.parent.mkdir(parents=True, exist_ok=True)
indicators.to_csv(DATA_CSV, index=False, encoding='utf-8-sig')
print(f'指标数据已保存: {DATA_CSV}')
indicators.tail()

## 5. 最近交易日摘要

In [ ]:
summary = latest_indicator_summary(indicators)
write_summary_markdown(summary, SUMMARY_MD)
summary

## 6. 生成图表

下面代码生成 SVG 图表，后面的 Markdown 单元格会直接展示这些图。

In [ ]:
figure_paths = write_all_svg_charts(indicators, FIGURE_DIR)
for path in figure_paths:
    print(path)

### 收盘价与布林带

![收盘价与布林带](../outputs/figures/price_bollinger.svg)

### 成交量

![成交量](../outputs/figures/volume.svg)

### RSI

![RSI](../outputs/figures/rsi.svg)

### MACD

![MACD](../outputs/figures/macd.svg)

### ATR

![ATR](../outputs/figures/atr.svg)

## 7. 简短结论和风险提示

技术指标显示的是历史价格、趋势动能和波动状态。RSI 用于观察短期强弱，MACD 用于观察趋势动能，布林带用于观察价格相对均值的位置，ATR 用于观察波动风险。以上结果不构成投资建议。